In [1]:
from tablevault import tablevault
import os
vault = tablevault.Vault(user_id="jinjin",
                            process_name="distilbert_feature_extraction_cosine_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
import torch
import numpy as np
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)



---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [4]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [5]:
model_name = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print('hidden_size:', model.config.hidden_size)



---[ TableVault Record ]---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768
---[ TableVault Record ]---



In [6]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds['sentence1']
sent2 = ds['sentence2']
y_true = np.array(ds['label'])

print('num_examples:', len(y_true))
print('positive_rate:', float(y_true.mean()))



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [7]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    pooled = summed / counts
    pooled = F.normalize(pooled, p=2, dim=1)
    return pooled


def encode_sentences(sentences, batch_size=128, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size)):
            batch = sentences[i:i + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            pooled = mean_pool(outputs.last_hidden_state, enc['attention_mask'])
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0)



---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
emb1 = encode_sentences(sent1)
emb2 = encode_sentences(sent2)

vault.create_embedding_list("distilbert-feature-embedding-sentence-1", ndim=768)
vault.create_embedding_list("distilbert-feature-embedding-sentence-2", ndim=768)

e1_list = emb1.tolist()
e2_list = emb2.tolist()

for i in range(len(e1_list)):
        vault.append_embedding("distilbert-feature-embedding-sentence-1", e1_list[i], 
                           input_items = {"glue_mrpc_validation": [i, i + 1]}
                           )
        vault.append_embedding("distilbert-feature-embedding-sentence-2", e2_list[i], 
                           input_items = {"glue_mrpc_validation": [i, i + 1]}
                           )
    
description = "distilbert-feature-embedding-sentence-1 is an embedding dataset containing one 768-dimensional dense vector for the sentence1 field of each example in the GLUE MRPC validation set. Each entry is produced by tokenizing the original sentence1 text with distilbert-base-uncased, running it through the DistilBERT encoder, applying attention-mask-aware mean pooling over the last hidden states, and L2-normalizing the pooled result. The dataset structure is an ordered embedding list: each item is a single float vector of length 768, and each vector is linked back to the corresponding row in glue_mrpc_validation through the input_items index mapping. In this workflow, it serves as the representation of the first sentence in each sentence pair and is used together with distilbert-feature-embedding-sentence-2 to compute cosine similarity scores for paraphrase prediction."
embedding = get_embeddings(description)
vault.create_description("distilbert-feature-embedding-sentence-1", description, embedding)

properties = {"task": "paraphrase detection", "representation": "text embedding", "embedding_model": "distilbert-base-uncased", "feature_extraction_method": "last_hidden_state_mean_pooling", "normalization": "l2", "embedding_dim": "768", "input_column": "sentence1", "paired_with": "sentence2", "source": "glue/mrpc", "split": "validation", "size": "408", "text_type": "sentence", "similarity_metric": "cosine similarity", "max_length": "128"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-feature-embedding-sentence-1", cat, embedding, prop)

description = "This dataset stores the DistilBERT feature embeddings for the sentence2 field of each example in the GLUE MRPC validation split. Each entry is a single 768-dimensional dense vector produced by encoding the raw sentence2 text with distilbert-base-uncased, applying mean pooling over the last hidden states with the attention mask, and L2-normalizing the result. The dataset is an embedding list rather than a tabular record set, so its primary field is the embedding vector itself; each embedding is linked by index to the corresponding source row in glue_mrpc_validation through input_items metadata. In this workflow, these sentence2 embeddings are paired with the matching sentence1 embeddings from distilbert-feature-embedding-sentence-1 to compute cosine similarity scores, which are then thresholded to generate paraphrase predictions."
embedding = get_embeddings(description)
vault.create_description("distilbert-feature-embedding-sentence-2", description, embedding)

properties = {"task": "paraphrase detection", "representation": "sentence embedding", "embedding_model": "distilbert-base-uncased", "embedding_dim": "768", "pooling": "mean pooling with L2 normalization", "input_field": "sentence2", "source_dataset": "glue/mrpc", "split": "validation", "text_type": "sentence pair component", "domain": "news"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-feature-embedding-sentence-2", cat, embedding, prop)



---[ TableVault Record ]---


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

---[ TableVault Record ]---



In [9]:
cosine_scores = (emb1 * emb2).sum(dim=1).numpy()
threshold = 0.85
y_pred = (cosine_scores >= threshold).astype(int)

print('done')
print('embedding_shape:', tuple(emb1.shape))
print('score_range:', float(cosine_scores.min()), float(cosine_scores.max()))

vault.create_record_list("distilbert-feature-embedding_prediction", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("distilbert-feature-embedding_prediction", {"prediction": int(y_pred[i])}, 
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                           "distilbert-feature-embedding-sentence-1": [i, i + 1],
                           "distilbert-feature-embedding-sentence-2": [i, i + 1],
                       }
                       )

description = "This dataset stores the binary paraphrase predictions produced for each example in the GLUE MRPC validation set using DistilBERT sentence embeddings. Each record corresponds to one sentence pair and contains a single field, prediction, where 1 indicates the pair was predicted to be a paraphrase and 0 indicates not paraphrase. The predictions are generated by computing the cosine similarity between the mean-pooled, L2-normalized embeddings of sentence1 and sentence2 and applying a fixed threshold of 0.85. In this workflow, this dataset serves as the model output table used for downstream evaluation, error analysis, and summary reporting, and each prediction record is linked back to the original MRPC validation example and the two embedding datasets for sentence1 and sentence2."
embedding = get_embeddings(description)
vault.create_description("distilbert-feature-embedding_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "prediction records", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "distilbert-base-uncased", "embedding_method": "DistilBERT mean-pooled sentence embeddings", "similarity_metric": "cosine similarity", "decision_rule": "threshold at 0.85", "label_type": "binary prediction", "input_pair": "sentence1 and sentence2", "domain": "sentence pairs"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-feature-embedding_prediction", cat, embedding, prop)


---[ TableVault Record ]---
done
embedding_shape: (408, 768)
score_range: 0.7408234477043152 0.9976521134376526
---[ TableVault Record ]---



In [10]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])
print({'accuracy': acc, 'f1': f1})
print(classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase']))



---[ TableVault Record ]---
{'accuracy': 0.696078431372549, 'f1': 0.8165680473372781}
                precision    recall  f1-score   support

not_paraphrase       0.73      0.06      0.11       129
    paraphrase       0.70      0.99      0.82       279

      accuracy                           0.70       408
     macro avg       0.71      0.53      0.47       408
  weighted avg       0.71      0.70      0.59       408

---[ TableVault Record ]---



In [11]:
for i in range(5):
    print('=' * 80)
    print('sentence1:', sent1[i])
    print('sentence2:', sent2[i])
    print('cosine:', float(cosine_scores[i]))
    print('true:', int(y_true[i]), 'pred:', int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print('num_errors:', int((y_true != y_pred).sum()))

for i in mistakes:
    print('=' * 80)
    print('idx:', int(i))
    print('sentence1:', sent1[i])
    print('sentence2:', sent2[i])
    print('cosine:', float(cosine_scores[i]))
    print('true:', int(y_true[i]), 'pred:', int(y_pred[i]))



---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
cosine: 0.9466804265975952
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
cosine: 0.8935593366622925
true: 0 pred: 1
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
cosine: 0.96232008934021
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announ

In [12]:
vault.create_record_list("distilbert_feature_extraction_cosine_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("distilbert_feature_extraction_cosine_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert-feature-embedding_prediction": [0, len(ds)]
                    })

summary

description = "Summary dataset for the DistilBERT cosine-similarity MRPC validation experiment. It contains a single record with aggregate evaluation metrics computed on the glue_mrpc_validation split after encoding sentence1 and sentence2 with distilbert-base-uncased, mean-pooling and L2-normalizing the token embeddings, computing cosine similarity between the two sentence embeddings, and converting scores to paraphrase predictions with a fixed threshold of 0.85. Fields: accuracy (float), f1 (float), and classification_report (string containing the per-class precision/recall/F1 summary for not_paraphrase and paraphrase). Its role in the workflow is to store the experiment-level performance summary linked to the full validation dataset and the per-example prediction dataset, so users can quickly inspect overall model quality without re-running the notebook."
embedding = get_embeddings(description)
vault.create_description("distilbert_feature_extraction_cosine_mrpc_summary", description, embedding)

properties = {"dataset_type": "evaluation_summary", "task": "paraphrase_detection", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "distilbert-base-uncased", "representation": "mean_pooled_sentence_embeddings", "embedding_dim": "768", "similarity_metric": "cosine_similarity", "decision_rule": "fixed_threshold", "threshold": "0.85", "label_space": "binary", "input_type": "sentence_pair", "metrics": "accuracy,f1,classification_report", "process_name": "distilbert_feature_extraction_cosine_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_feature_extraction_cosine_mrpc_summary", cat, embedding, prop)



---[ TableVault Record ]---
---[ TableVault Record ]---



In [13]:
description = "This notebook runs a paraphrase detection workflow on the GLUE MRPC validation set using distilbert-base-uncased as a feature extractor. It encodes each sentence in a pair with DistilBERT, applies mean pooling and L2 normalization to obtain 768-dimensional sentence embeddings, and predicts whether the pair is a paraphrase by thresholding the cosine similarity between the two embeddings. The notebook evaluates the resulting predictions with accuracy, F1, and a classification report, and prints example pairs and errors for inspection. Throughout the workflow, it logs the sentence embeddings, prediction records, summary metrics, and process metadata into TableVault/ArangoDB, and uses OpenAI text embeddings to store semantic descriptions of the generated artifacts." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("distilbert_feature_extraction_cosine_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "method": "feature extraction with cosine similarity thresholding", "model": "distilbert-base-uncased", "embedding_pooling": "mean pooling with L2 normalization", "dataset": "glue/mrpc", "dataset_split": "validation", "input_type": "sentence pair", "prediction_type": "binary classification", "similarity_metric": "cosine similarity", "threshold": "0.85", "framework": "transformers + pytorch", "embedding_storage": "tablevault", "evaluation": "accuracy, f1-score, classification report", "summary_artifact": "distilbert_feature_extraction_cosine_mrpc_summary", "process_name": "distilbert_feature_extraction_cosine_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_feature_extraction_cosine_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

